In [1]:
import os 
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models
from langchain_unstructured import UnstructuredLoader
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain.prompts import ChatPromptTemplate
from langchain_ollama import OllamaLLM
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client.http.models import Distance, VectorParams
from transformers import AutoTokenizer
from unstructured.cleaners.core import clean

In [2]:
FOLDER_PATH = './HPE Files/'
DB_PATH = './DB'

In [3]:
embed_model = "mxbai-embed-large"
llm_model = "llama3"
client = QdrantClient(path=DB_PATH)

In [4]:
prompt_template = """
Answer the question based only on the following context:

{context}

---

Answer the question based on the context above: {question}
"""

In [5]:
!ollama pull mxbai-embed-large

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 819c2adf5ce6... 100% ▕████████████████▏ 669 MB                         
pulling c71d239df917... 100% ▕████████████████▏  11 KB                         
pulling b837481ff855... 100% ▕████████████████▏   16 B                         
pulling 38badd946f91... 100% ▕████████████████▏  408 B                         
verifying sha256 digest 
writing manifest 
success 


In [6]:
!ollama pull llama3

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest 
pulling 6a0746a1ec1a... 100% ▕████████████████▏ 4.7 GB                         
pulling 4fa551d4f938... 100% ▕████████████████▏  12 KB                         
pulling 8ab4849b038c... 100% ▕████████████████▏  254 B                         
pulling 577073ffcc6c... 100% ▕████████████████▏  110 B                         
pulling 3f8eb4da87fa... 100% ▕████████████████▏  485 B                         
verifying sha256 digest 
writing manifest 
success 


In [7]:
def load_documents():
    docs = []
    for file in os.listdir(FOLDER_PATH):
        if file.endswith('.pdf'):
            pdf_path = FOLDER_PATH + "/" + file
            loader = UnstructuredLoader(pdf_path, partition_strategy='hi_res')
            docs.extend(loader.load())
    docs = filter_complex_metadata(docs)
    return docs

In [8]:
def split_documents(docs):
    size=800
    overlap=200
    tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
    splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=size, chunk_overlap=overlap)
    #splitter = RecursiveCharacterTextSplitter.(chunk_size=size, chunk_overlap=overlap)
    chunks = splitter.split_documents(docs)

    # custom: delete headers that just have filenames
    chunks = [ch for ch in chunks if ch.metadata.get("category") != 'Header']
    return chunks

In [9]:
def add_chunk_ids(chunks):
    last_page_id = None
    current_chunk_index = 0
    
    for chunk in chunks:
        source = chunk.metadata.get("filename")[:-7]
        page = chunk.metadata.get("page_number")
        current_page_id = f"{source}:{page}"

        if current_page_id == last_page_id:
            current_chunk_index += 1
        else:
            current_chunk_index = 0

        chunk_id = f"{current_page_id}:{current_chunk_index}"
        last_page_id = current_page_id
        chunk.metadata["id"] = chunk_id

    return chunks

In [10]:
def add_to_database(chunks):

    # create a collection if it doesn't exist already
    if not client.collection_exists("hpe_files"):
        client.create_collection(
        collection_name="hpe_files",
        vectors_config=VectorParams(size=1024, distance=Distance.COSINE))

    # initialize a vector score
    vector_store = QdrantVectorStore(
        client=client,
        collection_name="hpe_files",
        embedding=OllamaEmbeddings(model=embed_model))
    
    # get all the existing document ids in the collection
    existing_ids = []
    offset = ''
    while offset is not None:
        dop = client.scroll(collection_name='hpe_files', offset=offset)
        for dop_element in dop[0]:
            existing_ids.append(dop_element.payload.get("metadata").get("id"))
        offset=dop[1]

    # create chunk ids
    chunks_with_ids = add_chunk_ids(chunks)

    # filter for only new chunks
    new_chunks = []
    for chunk in chunks_with_ids:
        if chunk.metadata["id"] not in existing_ids:
            new_chunks.append(chunk)

    # add to the vector store only the new chunks
    if len(new_chunks):
        print(f"📄 Adding new document chunks: {len(new_chunks)}")
        vector_store.add_documents(new_chunks)
    else:
        print("✅ No new documents to add")

In [11]:
def delete_from_database():

    # get all existing files in the database
    existing_files = []
    offset = ''
    while offset is not None:
        dop = client.scroll(collection_name='hpe_files', offset=offset)
        for dop_element in dop[0]:
            file = dop_element.payload.get("metadata").get("filename")
            if file not in existing_files:
                existing_files.append(file)
        offset=dop[1]

    # get all files in the data folder
    data_files = os.listdir(FOLDER_PATH)

    # check which files are to be deleted
    to_delete = []
    for el in existing_files:
        if el not in data_files:
            to_delete.append(el)

    # if there are no items to delete, then do nothing
    if len(to_delete) == 0:
        print("✅ No documents to delete")
        return
    else:
    # delete points in the database if the files are not in the data folder
        print(f"❌ Deleting {len(to_delete)} file(s)")
        for delete in to_delete:
            client.delete(
            collection_name="hpe_files",
            points_selector=models.FilterSelector(
                filter=models.Filter(
                    must=[
                        models.FieldCondition(
                            key="metadata.filename",
                            match=models.MatchValue(value=delete))])))

In [12]:
def query_rag(query_text):
    db = QdrantVectorStore(
        client=client,
        collection_name="hpe_files",
        embedding=OllamaEmbeddings(model=embed_model))
    
    results = db.similarity_search_with_score(query_text, k=20)
    context_text = "\n\n---\n\n".join(["Product:" + docs.metadata.get("filename")[:-7] + "\n\n" + docs.page_content for docs, _score in results])
    #print(context_text)
    prompt = ChatPromptTemplate.from_template(prompt_template)
    prompt = prompt.format(context=context_text, question=query_text)
    
    model = OllamaLLM(model="llama3")
    response_text = model.invoke(prompt)
    formatted_response = f"Response: {response_text} \n\n"
    return response_text

In [13]:
docs = load_documents()
chunks = split_documents(docs)

INFO: NumExpr defaulting to 8 threads.
INFO: pikepdf C++ to Python logger bridge initialized


In [14]:
add_to_database(chunks)
delete_from_database()

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"


✅ No new documents to add
✅ No documents to delete


In [15]:
query = "Is the RTX 4000 GPU supported in any of HPE’s server models?" 
print(query_rag(query))

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


According to the provided context, the NVIDIA RTX 4000 Ada Graphics Accelerator is supported in the following HPE server models:

1. Product:HPE ProLiant ML350 Gen11-a50004308
2. Product:HPE ProLiant DL380 Gen11-a50004307 (only with certain configurations)

Note that the support for this GPU may have specific limitations or requirements, as mentioned in the context.


In [16]:
query = "How many GPUs can fit in a Proliant DL380a Gen11 server?"
print(query_rag(query))

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Based on the provided context, the HPE ProLiant DL380a Gen11 server supports:

* 4 double-wide accelerators (or)
* 8 single-wide accelerators

So, the answer is: up to 8 GPUs can fit in a Proliant DL380a Gen11 server.


In [17]:
query = "The customer has limited space in it’s datacenter and also no rack currently. Which server model would best fit its needs?"
print(query_rag(query))

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Based on the provided context, I recommend the HPE ProLiant ML350 Gen11-a50004308 as the best-fit server model for the customer's needs.

This is because:

1. The ML350 has a 4U tower form factor, which means it can be placed in a smaller space.
2. It also comes with an optional Tower-to-Rack conversion kit (P47394-B21) that allows it to be converted into a 5U rack-mount server if needed.
3. Since the customer does not currently have a rack, the ML350's tower form factor is a good fit.

In contrast, the HPE ProLiant DL360 Gen11-a50004306 is a standalone server and requires a rack for deployment. The HPE ProLiant DL325 Gen11-a50004297 is also a 1U rack-mount server, which may not be suitable for the customer's limited space.


In [18]:
query = "How many memory channels does the DL360 Gen 11 have per processor?" # added "per processor" wording
print(query_rag(query))

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


According to the context, the HPE ProLiant DL360 Gen11 has 8 channels per processor.


In [19]:
query = "Which CPU available for the DL380 Gen11 has the highest amount of cores, and how many does it have?"
print(query_rag(query))

INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/embed "HTTP/1.1 200 OK"
INFO: HTTP Request: POST http://127.0.0.1:11434/api/generate "HTTP/1.1 200 OK"


Based on the provided context, there is no mention of the HPE ProLiant DL380 Gen11 having a specific number of cores. The only mention of cores is in the description of the HPE ProLiant DL360 Gen10 Plus, which has up to 32 cores. Therefore, I cannot determine the exact number of cores for the HPE ProLiant DL380 Gen11 based on this information.
